# 02 · Evidencia de la ablación — panel de 30

Este notebook **no calcula ruido**. Carga los umbrales que produjo
`01_calibracion_ruido.ipynb` y los usa para decidir qué diferencias entre
condiciones son reales y cuáles caben dentro del azar.

La separación es deliberada: el umbral se fija *antes* y *sin mirar* las hipótesis.
Si el mismo notebook hiciera las dos cosas, se podría ajustar la estimación de ruido
hasta que diera el veredicto conveniente.

## Contenido

1. Carga de los umbrales del notebook 01
2. La tabla de ablación completa a n=30
3. ¿El rango entero cabe en el ruido?
4. **Comparaciones múltiples** — por qué 45 comparaciones rompen el p<0.05, y cómo se corrige
5. Comparaciones pareadas con corrección de Holm
6. Lo que el Edge F1 escondía: cantidad de aristas
7. El oráculo contra producción

In [1]:
import json
from itertools import combinations
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd

ABL = Path("../reports/ablation")
PISO = Path("../reports/piso_ruido_panel30.json")
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)

if not PISO.exists():
    raise SystemExit(
        "Falta results/piso_ruido_panel30.json — corré 01_calibracion_ruido.ipynb primero.\n"
        "Este notebook NO recalcula el ruido a propósito: hay una sola fuente de verdad."
    )

piso = json.loads(PISO.read_text(encoding="utf-8"))
MDE = {k: v["mde_30"] for k, v in piso["metricas"].items()}
SIGMA = {k: v["sigma_d"] for k, v in piso["metricas"].items()}

print(f"Umbrales cargados de {PISO.name} (generado {piso['generado'][:19]})")
print(f"Pares nulos usados para calibrar: {len(piso['pares_usados'])} "
      f"({piso['n_observaciones']} observaciones)\n")
display(pd.DataFrame({"sigma_d": SIGMA, "MDE n=30": MDE}).round(2))

Umbrales cargados de piso_ruido_panel30.json (generado 2026-08-30T17:45:28)
Pares nulos usados para calibrar: 3 (90 observaciones)



,sigma_d,MDE n=30
Edge F1,6.43,3.29
Service F1,3.45,1.76
aristas generadas,1.42,0.73
nodos generados,0.66,0.34
svc alucinados,0.41,0.21
svc faltantes,0.21,0.11


## 2 · La tabla de ablación a n=30

**Una condición es un prompt distinto**, identificado por el SHA-256 de su texto. Las
corridas que repiten el mismo SHA son réplicas y se promedian dentro de su condición:
si compitieran como filas separadas, el mismo prompt aparecería tres veces en la tabla
y las comparaciones por pares se inflarían de 55 a 91 sin agregar una sola hipótesis.

Se excluye del panel todo lo que **no sea un cambio de prompt**:

| se excluye | por qué |
|---|---|
| corridas **ensambladas** | resultados copiados de otras corridas, no generados de nuevo |
| **`connection_evidence`** | agrega evidencia al contexto — cambia la entrada |
| **oráculo** | reemplaza el world model de Stage 1 — cambia la entrada |
| **`transcript_enabled=False`** | quita la transcripción — cambia la entrada |
| brazo **Parsimonious** | otro pipeline, con otra Stage 1: la diferencia no sería del prompt |

Las últimas dos entraron a `reports/ablation/` el 30 y 31 de agosto, después de que
se fijaran las cifras de esta sección. Sin ellas en el filtro, el "peor prompt" del
panel pasaba a ser el pipeline del otro brazo corrido sin transcripción, y el rango
saltaba de 3.61 a 6.54 puntos sin que hubiera aparecido ninguna variante nueva.

In [2]:
import sys
sys.path.append("..")
from scripts.utils.effect_size import a12_ic, magnitud


def load_run(dirname: str) -> dict:
    return json.loads((ABL / dirname / "run.json").read_text(encoding="utf-8"))


def por_video(run: dict) -> dict:
    return {r["video_id"]: r for r in run["results"] if r.get("status") == "success"}


VALOR = {
    "Edge F1":           lambda r: 100 * r["edge_f1"],
    "Service F1":        lambda r: 100 * r["svc_f1"],
    "aristas generadas": lambda r: r["gen_edges"],
    "nodos generados":   lambda r: r["gen_nodes"],
    "svc alucinados":    lambda r: len(r.get("services_hallucinated") or []),
    "svc faltantes":     lambda r: len(r.get("services_missing") or []),
}


def es_prompt_puro(d: dict) -> bool:
    """¿Esta corrida difiere de producción SOLO en el texto del prompt de Stage 2?

    Cada `return False` de acá corresponde a una corrida que comparte prompt con
    producción pero recibió otra *entrada*. Mezclarlas en el panel confunde el efecto
    de cambiar el prompt con el de cambiar lo que el prompt tiene para leer, que es
    justo lo que esta sección intenta separar.
    """
    if d.get("connection_evidence_enabled"):
        return False
    if d.get("oracle"):
        return False
    if d.get("transcript_enabled") is False:
        return False
    # El brazo Parsimonious tiene su propia Stage 1; su comparación vive en el
    # notebook 10. No hay campo de brazo en run.json, así que se identifica por el
    # nombre del prompt, que es explícito.
    if "PARSIMONIOUS" in d["stage2_prompt"]["name"].upper():
        return False
    return True


# Un prompt = un SHA. Las réplicas del mismo SHA se acumulan y se promedian por video.
acum, etiquetas, replicas = {}, {}, {}
for p in sorted(ABL.glob("*/run.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    # reports/ablation/ tambien guarda corridas de otra naturaleza (la auditoria de
    # aristas de retorno, por ejemplo). Sin prompt de Stage 2 no son ablaciones.
    if "stage2_prompt" not in d:
        continue
    if any("source" in r for r in d["results"]):
        continue                                   # ensamblada
    if not es_prompt_puro(d):
        continue
    vids = por_video(d)
    if len(vids) < 30:
        continue
    sha = d["stage2_prompt"]["sha256"]
    etiquetas[sha] = f"{d['stage2_prompt']['name']} · {sha[:8]}"
    replicas[sha] = replicas.get(sha, 0) + 1
    for v, r in vids.items():
        acum.setdefault(sha, {}).setdefault(v, []).append(r)

condiciones = {}          # etiqueta -> {video: {metrica: valor promediado}}
for sha, por_v in acum.items():
    condiciones[etiquetas[sha]] = {
        v: {m: float(np.mean([f(r) for r in rs])) for m, f in VALOR.items()}
        for v, rs in por_v.items()
    }

filas = []
for etiqueta, vids in condiciones.items():
    fila = {"condicion": etiqueta,
            "réplicas": replicas[[s for s, e in etiquetas.items() if e == etiqueta][0]]}
    for m in VALOR:
        fila[m] = np.mean([r[m] for r in vids.values()])
    filas.append(fila)

tabla = pd.DataFrame(filas).set_index("condicion").sort_values("Edge F1", ascending=False)
n_corridas = sum(replicas.values())
print(f"{len(tabla)} prompts distintos (SHA-256), {n_corridas} corridas sobre los 30 videos")
print(f"comparaciones por pares: {len(tabla) * (len(tabla) - 1) // 2}\n")
display(tabla.round(2))

# El valor de las metricas del GT no depende de la condicion; se saca aparte.
GT_EDGES = np.mean([r["gt_edges"] for r in por_video(
    load_run(sorted(ABL.glob("*/run.json"))[0].parent.name)).values()]) \
    if False else None

11 prompts distintos (SHA-256), 14 corridas sobre los 30 videos
comparaciones por pares: 55



,réplicas,Edge F1,Service F1,aristas generadas,nodos generados,svc alucinados,svc faltantes
condicion,,,,,,,
STAGE2_V7_RETURN_FLOWS_V6 · 7ad8d936,1,60.10,88.07,9.73,8.90,0.73,1.00
STAGE2_V6_OPTIMIZED · 079b0aa8,1,59.14,86.75,8.97,9.00,0.87,1.03
STAGE2_V7_RETURN_FLOWS · 1ed4ebf9,1,58.65,86.91,10.17,9.23,0.93,0.90
STAGE2_V5_STRICT_ROUTING · a792c132,1,58.45,86.78,8.70,8.97,0.87,1.03
STAGE2_V8_ACTORS_AND_RETURNS · 277020ba,1,58.40,86.21,9.97,9.07,0.97,1.07
STAGE2_V6_CORRECTED · ed1d8505,4,58.13,87.10,9.03,8.89,0.82,1.03
STAGE2_V6_CORRECTED · dbddf1f3,1,58.10,88.47,9.47,9.10,0.77,0.87
STAGE2_V5_STRICT_ROUTING · 53f8d912,1,58.08,86.65,9.37,9.07,0.90,0.97
STAGE2_V6_CORRECTED · 7228956f,1,57.52,85.92,9.37,8.97,0.93,1.07


## 3 · ¿El rango entero cabe dentro del ruido?

La pregunta más directa: entre el mejor y el peor prompt de la tabla, ¿hay más
diferencia que la que produce repetir la misma llamada dos veces?

El MDE es lo que este diseño **podría** detectar. No es lo mismo que lo que
efectivamente detecta: eso lo decide la prueba pareada de la sección siguiente.

In [3]:
resumen = []
for m in VALOR:
    rango = tabla[m].max() - tabla[m].min()
    resumen.append({
        "metrica": m,
        "mejor": tabla[m].max(),
        "peor": tabla[m].min(),
        "rango": rango,
        "MDE n=30": MDE[m],
        "rango / MDE": rango / MDE[m],
        # Ojo con leer esto como "hay diferencia entre prompts". El MDE acota lo que el
        # diseño PODRIA detectar; que el rango lo exceda no implica que alguna
        # comparacion concreta sea detectable. La seccion 4 muestra que en Edge F1
        # ninguna lo es, pese a que el rango queda 1.10x por encima del umbral.
        "veredicto": ("el rango excede el umbral" if rango > MDE[m]
                      else "el rango entero cabe en el ruido"),
    })

display(pd.DataFrame(resumen).set_index("metrica").round(2))
print("El veredicto es sobre el RANGO, no sobre ninguna comparacion en particular:\n"
      "dice si el diseno tiene resolucion suficiente para que valga la pena mirar,\n"
      "no si hay una diferencia real. Eso lo decide la prueba pareada de la seccion 4.")

,mejor,peor,rango,MDE n=30,rango / MDE,veredicto
metrica,,,,,,
Edge F1,60.10,56.49,3.61,3.29,1.10,el rango excede el umbral
Service F1,88.47,85.62,2.84,1.76,1.61,el rango excede el umbral
aristas generadas,10.17,8.67,1.50,0.73,2.07,el rango excede el umbral
nodos generados,9.23,8.83,0.40,0.34,1.18,el rango excede el umbral
svc alucinados,1.03,0.73,0.30,0.21,1.42,el rango excede el umbral
svc faltantes,1.07,0.87,0.20,0.11,1.86,el rango excede el umbral


El veredicto es sobre el RANGO, no sobre ninguna comparacion en particular:
dice si el diseno tiene resolucion suficiente para que valga la pena mirar,
no si hay una diferencia real. Eso lo decide la prueba pareada de la seccion 4.


## 4 · Comparaciones por pares: p crudo, tamaño de efecto, y recién después corrección

### Por qué en este orden

Arcuri y Briand —la guía canónica para evaluar algoritmos aleatorizados en ingeniería
de software— **desaconsejan explícitamente el ajuste de Bonferroni**. Su argumento es
que resulta excesivamente conservador, infla el error de Tipo II, y en contextos donde
hay que elegir una técnica entre varias termina forzando decisiones arbitrarias bajo
una supuesta "no significancia". Lo que recomiendan en su lugar es reportar **todos los
p-valores crudos**, acompañados del **tamaño de efecto con su intervalo de confianza**.

Por eso esta sección lidera con eso y deja la corrección como control de robustez. La
conclusión de RQ2 no depende de Bonferroni y no debería parecer que sí: el resultado
central es que la comparación extremo-contra-extremo **ya no es significativa sin
corregir**, y que los tamaños de efecto son insignificantes en toda la tabla.

### El tamaño de efecto que corresponde

**Â₁₂ de Vargha y Delaney**, que es el que Arcuri recomienda para escala de intervalo.
Mide la probabilidad de que un video tomado al azar puntúe mejor con el prompt A que
con el B, contando empates como medio. **0.5 es ausencia de efecto**; los umbrales de
magnitud son 0.56 (pequeño), 0.64 (mediano) y 0.71 (grande).

Se descarta la familia de la *d* de Cohen por la razón que dan los mismos autores:
asume normalidad, y las distribuciones de F1 por video de este proyecto son
marcadamente asimétricas —una masa de videos en 100 y una cola larga hacia 0—.

### Y para qué sirve el intervalo de confianza

Es lo que distingue **"no hay efecto"** de **"no hay potencia"** ante un resultado no
significativo. Un IC angosto centrado en 0.5 es evidencia *de ausencia* de efecto; uno
ancho es simplemente falta de potencia. Esa distinción es la afirmación exacta que
sostiene RQ2 — *ninguna diferencia es detectable* — y sin ella el resultado negativo no
se puede defender.

### La corrección, como control secundario

Con $k$ prompts hay $\binom{k}{2}$ comparaciones, y la probabilidad de al menos un
falso positivo sin corregir es $1 - (1-\alpha)^{\binom{k}{2}}$. Se reporta Holm y
Bonferroni para mostrar que la conclusión **tampoco cambia** bajo el criterio más
conservador — no porque la conclusión dependa de ellos.

In [4]:
def test_signos(a: dict, b: dict, m: str) -> tuple[int, int, int, float]:
    """Cuantos videos gana A sobre B. p-valor binomial exacto a dos colas."""
    comunes = sorted(set(a) & set(b))
    gana = sum(1 for v in comunes if a[v][m] > b[v][m])
    pierde = sum(1 for v in comunes if a[v][m] < b[v][m])
    empata = len(comunes) - gana - pierde
    n = gana + pierde
    if n == 0:
        return gana, pierde, empata, 1.0
    cola = sum(comb(n, k) for k in range(min(gana, pierde) + 1))
    return gana, pierde, empata, min(1.0, 2 * cola / 2**n)


def holm(pvals: list[float], alfa: float = 0.05) -> list[bool]:
    """Holm-Bonferroni. Devuelve, en el orden original, si cada hipotesis se rechaza.

    Frena en la primera que no pasa su umbral: todas las siguientes quedan sin
    rechazar, aunque su p individual sea chico. Eso es lo que preserva la garantia.
    """
    m = len(pvals)
    orden = sorted(range(m), key=lambda i: pvals[i])
    rechaza = [False] * m
    for rango, i in enumerate(orden):
        if pvals[i] <= alfa / (m - rango):
            rechaza[i] = True
        else:
            break
    return rechaza


METRICA = "Edge F1"

comparaciones = []
for A, B in combinations(condiciones, 2):
    g, pd_, e, p = test_signos(condiciones[A], condiciones[B], METRICA)
    com = sorted(set(condiciones[A]) & set(condiciones[B]))
    xs = [condiciones[A][v][METRICA] for v in com]
    ys = [condiciones[B][v][METRICA] for v in com]
    a12, (lo, hi) = a12_ic(xs, ys, n_boot=4000, seed=0)
    comparaciones.append({"A": A.split(" · ")[0][:22], "B": B.split(" · ")[0][:22],
                          "dif medias": np.mean(xs) - np.mean(ys),
                          "gana A": g, "gana B": pd_, "empata": e,
                          "p crudo": p, "Â₁₂": a12, "IC bajo": lo, "IC alto": hi,
                          "magnitud": magnitud(a12), "IC incluye 0.5": lo <= 0.5 <= hi})

comp = pd.DataFrame(comparaciones)
m = len(comp)
comp["signif. sin corregir"] = comp["p crudo"] < 0.05
comp["signif. Bonferroni"] = comp["p crudo"] < 0.05 / m
comp["signif. Holm"] = holm(comp["p crudo"].tolist())
comp = comp.sort_values("p crudo")

print(f"metrica: {METRICA}   ·   {m} comparaciones entre {len(condiciones)} prompts distintos\n")
print("— lo que Arcuri y Briand piden que se reporte primero —")
print(f"  significativas SIN corregir       : {comp['signif. sin corregir'].sum()}/{m}")
print(f"  Â₁₂ rango                          : {comp['Â₁₂'].min():.3f} – {comp['Â₁₂'].max():.3f}")
print(f"  magnitudes de efecto              : {comp['magnitud'].value_counts().to_dict()}")
print(f"  IC de Â₁₂ que incluye 0.5          : {int(comp['IC incluye 0.5'].sum())}/{m}")
print(f"  ancho medio del IC                : {(comp['IC alto'] - comp['IC bajo']).mean():.3f}")
print("\n— control de robustez, no el argumento —")
print(f"  P(al menos un falso positivo sin corregir) = {1 - 0.95**m:.1%}")
print(f"  significativas con Bonferroni     : {comp['signif. Bonferroni'].sum()}   (umbral p < {0.05/m:.5f})")
print(f"  significativas con Holm           : {comp['signif. Holm'].sum()}\n")
display(comp.head(12).drop(columns=["signif. Bonferroni"]).round(4))

metrica: Edge F1   ·   55 comparaciones entre 11 prompts distintos

— lo que Arcuri y Briand piden que se reporte primero —
  significativas SIN corregir       : 3/55
  Â₁₂ rango                          : 0.452 – 0.553
  magnitudes de efecto              : {'insignificante': 55}
  IC de Â₁₂ que incluye 0.5          : 55/55
  ancho medio del IC                : 0.110

— control de robustez, no el argumento —
  P(al menos un falso positivo sin corregir) = 94.0%
  significativas con Bonferroni     : 0   (umbral p < 0.00091)
  significativas con Holm           : 0



,A,B,dif medias,gana A,gana B,empata,p crudo,Â₁₂,IC bajo,IC alto,magnitud,IC incluye 0.5,signif. sin corregir,signif. Holm
46,STAGE2_V6_OPTIMIZED,STAGE2_V6_CORRECTED,1.6211,13,2,15,0.0074,0.5239,0.4950,0.5633,insignificante,True,True,False
49,STAGE2_V7_RETURN_FLOWS,STAGE2_V6_CORRECTED,2.5844,15,3,12,0.0075,0.5456,0.4994,0.6061,insignificante,True,True,False
6,STAGE2_V4_ANTI_HALLUCI,STAGE2_V7_RETURN_FLOWS,-2.9997,3,13,14,0.0213,0.4517,0.3772,0.5150,insignificante,True,True,False
5,STAGE2_V4_ANTI_HALLUCI,STAGE2_V6_OPTIMIZED,-2.0363,3,11,16,0.0574,0.4728,0.4244,0.5133,insignificante,True,False,False
42,STAGE2_V5_STRICT_ROUTI,STAGE2_V6_CORRECTED,0.9392,10,3,17,0.0923,0.5228,0.4889,0.5672,insignificante,True,False,False
23,STAGE2_V5_STRICT_ROUTI,STAGE2_V7_RETURN_FLOWS,-2.0188,5,13,12,0.0963,0.4667,0.4044,0.5256,insignificante,True,False,False
50,STAGE2_V7_RETURN_FLOWS,STAGE2_V6_CORRECTED,2.0010,11,4,15,0.1185,0.5272,0.4911,0.5700,insignificante,True,False,False
3,STAGE2_V4_ANTI_HALLUCI,STAGE2_V8_ACTORS_AND_R,-1.3025,4,10,16,0.1796,0.4794,0.4161,0.5333,insignificante,True,False,False
4,STAGE2_V4_ANTI_HALLUCI,STAGE2_V5_STRICT_ROUTI,-1.3544,2,7,21,0.1797,0.4767,0.4311,0.5122,insignificante,True,False,False
22,STAGE2_V5_STRICT_ROUTI,STAGE2_V6_OPTIMIZED,-1.0555,5,11,14,0.2101,0.4894,0.4350,0.5417,insignificante,True,False,False


## 6 · Lo que el Edge F1 escondía

El Edge F1 de todos los prompts es indistinguible. Pero **cuántas aristas dibuja cada
uno** sí se separa del ruido — y eso es un hallazgo real que la métrica agregada tapa.

Acá es donde se ve por qué el notebook 01 tuvo que calibrar *cada* métrica: el umbral
de las aristas es otro, mucho más chico, y comparar contra el MDE del F1 habría dado
un veredicto equivocado.

In [5]:
vista = tabla[["Edge F1", "aristas generadas", "svc alucinados", "svc faltantes"]].copy()
display(vista.sort_values("aristas generadas", ascending=False).round(2))

rango_ar = tabla["aristas generadas"].max() - tabla["aristas generadas"].min()
rango_f1 = tabla["Edge F1"].max() - tabla["Edge F1"].min()
print(f"\nEdge F1          : rango {rango_f1:.2f}  vs MDE {MDE['Edge F1']:.2f}  "
      f"-> {rango_f1 / MDE['Edge F1']:.2f}x el umbral")
print(f"aristas generadas: rango {rango_ar:.2f}  vs MDE {MDE['aristas generadas']:.2f}  "
      f"-> {rango_ar / MDE['aristas generadas']:.2f}x el umbral")

# El mismo criterio de tamaño de efecto que en la seccion anterior, aplicado a la
# metrica que si se separa del ruido.
mejor = tabla["aristas generadas"].idxmax()
peor = tabla["aristas generadas"].idxmin()
com = sorted(set(condiciones[mejor]) & set(condiciones[peor]))
a12_ar, ic_ar = a12_ic([condiciones[mejor][v]["aristas generadas"] for v in com],
                       [condiciones[peor][v]["aristas generadas"] for v in com], seed=0)
print(f"\naristas, extremo vs extremo: Â₁₂ = {a12_ar:.3f} [{ic_ar[0]:.3f}, {ic_ar[1]:.3f}] "
      f"· {magnitud(a12_ar)}")

,Edge F1,aristas generadas,svc alucinados,svc faltantes
condicion,,,,
STAGE2_V7_RETURN_FLOWS · 1ed4ebf9,58.65,10.17,0.93,0.90
STAGE2_V8_ACTORS_AND_RETURNS · 277020ba,58.40,9.97,0.97,1.07
STAGE2_V7_RETURN_FLOWS_V6 · 7ad8d936,60.10,9.73,0.73,1.00
STAGE2_V6_CORRECTED · dbddf1f3,58.10,9.47,0.77,0.87
STAGE2_V6_CORRECTED · 7228956f,57.52,9.37,0.93,1.07
STAGE2_V5_STRICT_ROUTING · 53f8d912,58.08,9.37,0.90,0.97
STAGE2_V4_DYNAMIC_FEW_SHOT · 45da674e,56.49,9.33,0.77,0.90
STAGE2_V6_CORRECTED · ed1d8505,58.13,9.03,0.82,1.03
STAGE2_V6_OPTIMIZED · 079b0aa8,59.14,8.97,0.87,1.03



Edge F1          : rango 3.61  vs MDE 3.29  -> 1.10x el umbral
aristas generadas: rango 1.50  vs MDE 0.73  -> 2.07x el umbral



aristas, extremo vs extremo: Â₁₂ = 0.622 [0.558, 0.700] · pequeño


## 7 · El oráculo contra producción

Esta comparación **no paga peaje de comparaciones múltiples**: es una hipótesis única,
planteada antes de correr el experimento, no una pesca entre pares.

In [6]:
# Las CUATRO replicas de produccion del panel de 30 con el prompt de produccion
# (sha ed1d8505), sin oraculo y con transcript. Es el conjunto completo: una quinta
# corrida del mismo prompt existe pero es del panel de 14 y no corresponde aca.
PROD = ["2026-08-25_1637_STAGE2_V6_CORRECTED_cell9_p30",
        "2026-08-27_1640_STAGE2_V6_CORRECTED_cell9_p30_rep2",
        "2026-08-29_0442_STAGE2_V6_CORRECTED_cell9_p30_rep3",
        "2026-08-29_0715_STAGE2_V6_CORRECTED_cell9_p30_rep4"]
ORAC = ["2026-08-28_0315_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt",
        "2026-08-28_2211_STAGE2_V6_CORRECTED_cell9_p30_oracle-oracle_gt_rep2"]


def promedio_replicas(dirs, f):
    acum = {}
    for d in dirs:
        for v, r in por_video(load_run(d)).items():
            acum.setdefault(v, []).append(f(r))
    return {v: float(np.mean(xs)) for v, xs in acum.items()}


filas = []
for m, f in VALOR.items():
    prod, orac = promedio_replicas(PROD, f), promedio_replicas(ORAC, f)
    comunes = sorted(set(prod) & set(orac))
    p_mean = np.mean([prod[v] for v in comunes])
    o_mean = np.mean([orac[v] for v in comunes])
    a12_o, ic_o = a12_ic([orac[v] for v in comunes], [prod[v] for v in comunes], seed=0)
    fila = {"metrica": m, "produccion": p_mean, "oraculo": o_mean,
            "delta": o_mean - p_mean}
    if m in MDE:
        fila["MDE n=30"] = MDE[m]
        fila["veces el MDE"] = (o_mean - p_mean) / MDE[m]
    fila["Â₁₂"] = a12_o
    fila["IC 95%"] = f"[{ic_o[0]:.3f}, {ic_o[1]:.3f}]"
    fila["magnitud"] = magnitud(a12_o)
    filas.append(fila)

print(f"Produccion promedia {len(PROD)} replicas, oraculo {len(ORAC)}; "
      f"ambas sobre los mismos 30 videos.")
print("Â₁₂ > 0.5 significa que el oraculo gana; es una hipotesis unica y "
      "preespecificada,\nasi que no paga peaje de comparaciones multiples.\n")
display(pd.DataFrame(filas).set_index("metrica").round(3))

Produccion promedia 2 replicas, oraculo 2; ambas sobre los mismos 30 videos.
Â₁₂ > 0.5 significa que el oraculo gana; es una hipotesis unica y preespecificada,
asi que no paga peaje de comparaciones multiples.



,produccion,oraculo,delta,MDE n=30,veces el MDE,Â₁₂,IC 95%,magnitud
metrica,,,,,,,,
Edge F1,58.425,82.172,23.747,3.288,7.223,0.856,"[0.788, 0.924]",grande
Service F1,87.549,99.569,12.020,1.764,6.813,0.906,"[0.829, 0.975]",grande
aristas generadas,9.067,9.967,0.900,0.726,1.239,0.597,"[0.522, 0.682]",pequeño
nodos generados,8.917,8.733,-0.183,0.339,-0.541,0.479,"[0.403, 0.552]",insignificante
svc alucinados,0.800,0.017,-0.783,0.212,-3.697,0.155,"[0.071, 0.241]",grande
svc faltantes,1.000,0.033,-0.967,0.108,-8.970,0.178,"[0.092, 0.267]",grande


## Qué se puede afirmar, y qué no

**Sí se puede afirmar:**

- **Ninguna diferencia entre prompts es detectable.** La comparación extremo contra
  extremo no alcanza significancia *sin corregir*, los tamaños de efecto Â₁₂ son
  insignificantes en toda la tabla, y sus intervalos de confianza son angostos y
  rodean el 0.5 — que es la forma que recomiendan Arcuri y Briand de distinguir
  ausencia de efecto de falta de potencia.
- El oráculo mejora Edge F1 y Service F1 muy por encima del MDE, con el test de signos
  y el tamaño de efecto coincidiendo. Es una hipótesis única y preespecificada.
- Los prompts difieren de forma detectable en **cuántas aristas dibujan**, aunque no en
  Edge F1. La métrica agregada tapaba ese efecto.
- Todos los prompts **subdibujan** aristas respecto del ground truth.

**No se puede afirmar:**

- Que un prompt de la tabla sea mejor que otro en Edge F1.
- Que las variantes sean *equivalentes*. La afirmación defendible es la más débil:
  con este diseño ninguna diferencia es detectable. El IC angosto alrededor de 0.5
  acota cuánto podría estar escondido, y no lo lleva a cero.
- Nada que dependa de una sola corrida por condición. De los 11 prompts, sólo uno
  tiene réplicas; los otros diez descansan en una corrida cada uno, y su posición
  relativa en la tabla no es estable a nivel de una repetición.

**Sobre el número de repeticiones.** Arcuri y Briand recomiendan del orden de 1000
corridas por condición, y GenProg reporta 100 por programa. Este trabajo tiene entre 1
y 4. La diferencia no es de criterio sino de costo: los dos precedentes justifican su
número observando que una corrida sólo consume CPU, mientras que acá cada corrida son
30 llamadas a una API comercial con cuota de 20 por día en el tier gratuito. Reproducir
el panel diez veces son 300 llamadas y varios días. La cota es real y acota lo que este
diseño puede afirmar; se declara en lugar de disimularla.